In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('ggplot')

In [ ]:
PROPS = pd.read_csv("../output/validation/props.csv", index_col="run_id")
VARIATION_INDICES = [
    [0, 10], [10, 20], [20, 30]
]
PROPS


In [ ]:
frames = [pd.read_csv("../output/validation/dynamic_results.csv")]
for i in range(2, 1000):
    try:
        frames.append(pd.read_csv(f"../output/validation/dynamic_results_{i}.csv"))
    except FileNotFoundError:
        print(f"Stopping at {i}")
        break
SIM_DF_RAW = pd.concat(frames)

SIM_DFS = [SIM_DF_RAW[(SIM_DF_RAW.run_id >= start) & (SIM_DF_RAW.run_id < end)].groupby("tick").mean() for start, end in VARIATION_INDICES]
SIM_DFS[0]

In [ ]:
total_lines = 725

PLOT_LIMITS = [
    {"start": 18, "end": 188},
    {"start": 198, "end": 367},
    {"start": 377, "end": 546},
    {"start": 556, "end": total_lines},
]
PLOT_COLUMNS = [
    {"y": "total_believers", "y.1": "total_susceptibles", "y.2": "total_fact_checkers"},
    {"y": "normal_believers", "y.1": "normal_susceptibles", "y.2": "normal_fact_checkers"},
    {"y": "scholar_believers", "y.1": "scholar_susceptibles", "y.2": "scholar_fact_checkers"},
    {"y": "influencer_believers", "y.1": "influencer_susceptibles", "y.2": "influencer_fact_checkers"},
]

frames = []

for i in range(1000):
    if i == 2:
        continue
    try:
        inner_frames = []
        for j in range(4):
            df = pd.read_csv(f"../validation/run{i}.csv", skiprows=PLOT_LIMITS[j]["start"], nrows=PLOT_LIMITS[j]["end"] - PLOT_LIMITS[j]["start"] - 1)
            df = df.filter(["x", "y", "y.1", "y.2"], axis=1)
            df = df.convert_dtypes(convert_integer=True)
            df = df.rename({"x": "tick", **PLOT_COLUMNS[j]}, axis=1)
            df = df.set_index("tick")
            inner_frames.append(df)
        total_frame = pd.concat(inner_frames, axis=1, join="inner")
        total_frame["run_id"] = i
        frames.append(total_frame)
    except FileNotFoundError:
        print(f"Stopped at {i}")
        break

REAL_DF_RAW = pd.concat(frames)
REAL_DFS = [REAL_DF_RAW[(REAL_DF_RAW.run_id >= start) & (REAL_DF_RAW.run_id < end)].groupby("tick").mean() for start, end in VARIATION_INDICES]
REAL_DFS[0]

In [ ]:
for i in range(len(VARIATION_INDICES)):
    plt.figure(figsize=(16, 9))
    plt.plot(SIM_DFS[i].total_believers, color="orange", label="[RepastHPC] Total believers")
    plt.plot(SIM_DFS[i].total_fact_checkers, color="blue", label="[RepastHPC] Total fact-checkers")
    plt.plot(SIM_DFS[i].total_susceptibles, color="green", label="[RepastHPC] Total susceptibles")
    plt.plot(REAL_DFS[i].total_believers, color="orange", label="[NetLogo] Total believers", marker=".", linewidth=0)
    plt.plot(REAL_DFS[i].total_fact_checkers, color="blue", label="[NetLogo] Total fact-checkers", marker=".", linewidth=0)
    plt.plot(REAL_DFS[i].total_susceptibles, color="green", label="[NetLogo] Total susceptibles", marker=".", linewidth=0)
    plt.title('Believers vs FactCheckers vs Susceptibles over time')
    plt.xlabel('Tick')
    plt.ylabel('Agent count')
    plt.legend()
    plt.show()
